# TempDiff-US on Beltsville

The TempDiff-SMID rule (*Agronomy* 2025, 15(6), 1438): silage is harvested earlier, so on a date when silage is cut and grain still stands, a silage field reads bare and dry (low LSWI).

**Fixed before scoring, without Beltsville labels:** date September 28 (USDA Usual Planting and Harvesting Dates, Maryland: middle between the corn-silage and corn-grain most-active harvest windows) and LSWI below 0.05 (FieldVision's bare-soil level). Scored once on all 44 Beltsville field-years. Free: no Nemotron calls.

In [ ]:
# 1. Setup
import os, sys, json
from fieldvision_agents.settings import REPO; os.chdir(REPO)
from fieldvision_agents.tempdiff import tempdiff as td
print(json.dumps(td.RULE, indent=1))

In [ ]:
# 2. Offline tests (synthetic seasons)
from fieldvision_agents.tempdiff import test_tempdiff
test_tempdiff.run_all()

In [ ]:
# 3. Score once on Beltsville (44 field-years, 2021-2023), pre-declared settings
df, m, by_year, sens = td.run('data/counties/24033', 'runs/tempdiff')
print('ALL:', json.dumps(m, indent=1))
for y, v in by_year.items():
    print(y, f"{v['correct']} of {v['n']} correct", '| silage recall', v['silage_recall'], '| FPR', v['fpr'])
print('\nFrom the model cards (same 44 field-years): eu_logistic_v1 42 of 44, eu_timing_rule_v1 38 of 44')

In [ ]:
# 4. Field by field: where it is right and wrong
import pandas as pd
pd.set_option('display.width', 200)
df.assign(ok=df['call'].fillna('grain') == df['label'])[['field_id', 'year', 'label', 'call', 'ok', 'lswi_on_date', 'ndvi_on_date', 'nearest_obs_days', 'low_evidence']]

In [ ]:
# 5. Sensitivity (post hoc, NOT a model choice): how much the result depends on the two fixed settings
sens.pivot(index='date', columns='lswi_threshold', values='correct')